# 03 — Fetch Author Information via OpenAlex

This notebook shows how to:
1. Extract author names from the example paper dataset
2. Search OpenAlex for an author by name
3. Retrieve their profile: ORCID, institution, h-index, citation count
4. Handle ambiguity when multiple authors share the same name

**API docs:** https://docs.openalex.org/api-entities/authors

In [1]:
import requests
import pandas as pd
import json

In [2]:
MAILTO = 'your.email@example.com'
BASE_URL = 'https://api.openalex.org/authors'

## 1. Extract author names from the example dataset

In [3]:
df = pd.read_csv('../data/example_papers.csv')

# Parse first author from each paper
def get_first_author(authors_str):
    if pd.isna(authors_str):
        return None
    # Authors are formatted as "Last, First; Last, First; ..."
    first = authors_str.split(';')[0].strip()
    # Reformat to "First Last"
    parts = first.split(',')
    if len(parts) == 2:
        return f'{parts[1].strip()} {parts[0].strip()}'
    return first

df['first_author_name'] = df['Authors'].apply(get_first_author)
df[['Title', 'first_author_name']].head(6)

,Title,first_author_name
0,An Efficient Machine Learning-Based Emotional ...,Lamiaa Abdel-Hamid
1,Machine learning explainability in nasopharyng...,Rasheed Omobolaji Alabi
2,Machine learning algorithms for prediction of ...,Omar M Fahmy
3,Machine learning for predicting neurodegenerat...,Gloria A. Aguayo
4,Explanatory predictive model for COVID-19 seve...,Mariam Laatifi
5,Monitoring Cardiovascular Problems in Heart Pa...,Ahmed Al Ahdal


## 2. Search for an author by name

In [4]:
author_name = df['first_author_name'].iloc[0]
print('Searching for:', author_name)

params = {
    'search': author_name,
    'per_page': 5,
    'mailto': MAILTO,
}

response = requests.get(BASE_URL, params=params)
data = response.json()

print(f"\nFound {data['meta']['count']} matching authors")

Searching for: Lamiaa Abdel-Hamid

Found 3 matching authors


In [5]:
# Display the candidates
for i, author in enumerate(data['results'], 1):
    institution = ''
    if author.get('last_known_institution'):
        institution = author['last_known_institution'].get('display_name', '')
    print(f"{i}. {author['display_name']}")
    print(f"   ID          : {author['id']}")
    print(f"   ORCID       : {author.get('orcid', 'N/A')}")
    print(f"   Institution : {institution}")
    print(f"   Works count : {author.get('works_count', 0)}")
    print(f"   Citations   : {author.get('cited_by_count', 0)}")
    print()

1. Lamiaa Abdel‐Hamid
   ID          : https://openalex.org/A5055186980
   ORCID       : https://orcid.org/0000-0002-4066-6021
   Institution : 
   Works count : 18
   Citations   : 360

2. Lamiaa Abdel-Hamid
   ID          : https://openalex.org/A5000777037
   ORCID       : None
   Institution : 
   Works count : 2
   Citations   : 5

3. Lamiaa Abdel-Hamid
   ID          : https://openalex.org/A5121774812
   ORCID       : None
   Institution : 
   Works count : 1
   Citations   : 0



## 3. Narrow down ambiguous results

If multiple authors share the same name, you can filter by institution or by a known work DOI.

In [6]:
# Strategy 1: pick the author with the most works (most likely to be the active researcher)
candidates = data['results']
if candidates:
    best_match = max(candidates, key=lambda a: a.get('works_count', 0))
    print('Best match by works count:')
    print(f"  Name       : {best_match['display_name']}")
    print(f"  OpenAlex ID: {best_match['id']}")
    print(f"  ORCID      : {best_match.get('orcid', 'N/A')}")
    print(f"  Works      : {best_match.get('works_count')}")
    print(f"  Citations  : {best_match.get('cited_by_count')}")

Best match by works count:
  Name       : Lamiaa Abdel‐Hamid
  OpenAlex ID: https://openalex.org/A5055186980
  ORCID      : https://orcid.org/0000-0002-4066-6021
  Works      : 18
  Citations  : 360


In [7]:
# Strategy 2: look up an author by a specific paper DOI
paper_doi = df['DOI'].iloc[0]
url = f'https://api.openalex.org/works/https://doi.org/{paper_doi}'
work = requests.get(url, params={'mailto': MAILTO}).json()

# Get the first author's OpenAlex ID directly from the work
if work.get('authorships'):
    first_authorship = work['authorships'][0]
    author_from_work = first_authorship.get('author', {})
    print('Author resolved via paper DOI:')
    print(f"  Name       : {author_from_work.get('display_name')}")
    print(f"  OpenAlex ID: {author_from_work.get('id')}")
    print(f"  ORCID      : {author_from_work.get('orcid', 'N/A')}")

Author resolved via paper DOI:
  Name       : Lamiaa Abdel‐Hamid
  OpenAlex ID: https://openalex.org/A5055186980
  ORCID      : https://orcid.org/0000-0002-4066-6021


## 4. Fetch full author profile by OpenAlex ID

In [8]:
author_id = author_from_work['id']  # e.g. 'https://openalex.org/A1234567890'

author_response = requests.get(author_id, params={'mailto': MAILTO})
author = author_response.json()

print('Full author profile:')
print(json.dumps({
    'display_name': author.get('display_name'),
    'orcid': author.get('orcid'),
    'works_count': author.get('works_count'),
    'cited_by_count': author.get('cited_by_count'),
    'h_index': author.get('summary_stats', {}).get('h_index'),
    'i10_index': author.get('summary_stats', {}).get('i10_index'),
    'last_known_institution': author.get('last_known_institution', {}).get('display_name'),
    'country_code': author.get('last_known_institution', {}).get('country_code'),
}, indent=2))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [9]:
# Publications per year (counts_by_year)
counts_by_year = author.get('counts_by_year', [])
if counts_by_year:
    df_years = pd.DataFrame(counts_by_year)[['year', 'works_count', 'cited_by_count']]
    print('Publication activity by year:')
    print(df_years.sort_values('year').to_string(index=False))

Publication activity by year:
 year  works_count  cited_by_count
 2026            1               0
